# ETL Step

In [6]:
import pandas as pd 
import numpy as np
from pathlib import Path
from IPython.display import display

json_files = list(Path('data').glob('*.json'))
dfs = []

COLUMNS_TO_KEEP = [
    'ts',
    'platform',
    'ms_played',
    'conn_country',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'reason_start',
    'reason_end',
    'shuffle',
    'skipped',
    'offline'
]

for file in json_files:
    try:
        dfs.append(pd.read_json(file)[COLUMNS_TO_KEEP])
        print(f"Loaded {file.name} successfully.")
    except Exception as e:
        print(f"Error loading {file}: {e}")

df = pd.concat(dfs, ignore_index=True)

COLUMN_RENAME = {
    'ts': 'Timestamp',
    'platform': 'Platform',
    'ms_played': 'Time Played (ms)',
    'conn_country': 'Country',
    'master_metadata_album_artist_name': 'Artist Name',
    'master_metadata_album_album_name': 'Album Name',
    'reason_start': 'Reason Started',
    'reason_end': 'Reason Ended',
    'shuffle': 'Shuffle',
    'skipped': 'Skipped',
    'offline': 'Offline'
}
df.rename(columns=COLUMN_RENAME, inplace=True)

df['Platform Main'] = (
    df['Platform']
    .str.split()
    .str[0]
    .replace({
        'Partner': 'Android TV',
        'web_player': 'Web Player',
        'WebPlayer': 'Web Player'
    })
    .str.upper()    
)

df['Length_min'] = df['Time Played (ms)'] / 60000
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df['Year'] = df['Timestamp'].dt.year


Loaded Streaming_History_Audio_2014-2019_0.json successfully.
Loaded Streaming_History_Audio_2019-2020_1.json successfully.
Loaded Streaming_History_Audio_2020-2021_2.json successfully.
Loaded Streaming_History_Audio_2021-2022_3.json successfully.
Loaded Streaming_History_Audio_2022-2023_4.json successfully.
Loaded Streaming_History_Audio_2023-2024_5.json successfully.
Loaded Streaming_History_Audio_2024-2025_6.json successfully.
Loaded Streaming_History_Audio_2025-2026_9.json successfully.
Loaded Streaming_History_Audio_2025_7.json successfully.
Loaded Streaming_History_Audio_2025_8.json successfully.
Loaded Streaming_History_Video_2020-2026.json successfully.


# Platform Analysis
This section will anser to question which platform was utilized the most by me

I was quite surprised by the total amount of time I have spent on Spotify.
That's why I converted the listening time up to years to better understand the scale.

In [ ]:
total_min = df['Length_min'].sum()

platform_pivot = df.pivot_table(
    index='Platform Main',
    values='Length_min', 
    aggfunc='sum'
    ).sort_values(by='Length_min', ascending=False)

platform_pivot = platform_pivot.assign(
    Length_hours=lambda x: x['Length_min'] / 60,
    Length_days=lambda x: x['Length_min'] / 1440,
    Length_months=lambda x: x['Length_min'] / 43200,
    Length_years=lambda x: x['Length_min'] / 525600,
    Pct=lambda x: (x['Length_min'] / total_min * 100)
)

display(platform_pivot.round(2))

years_pivot = df.pivot_table(
    index='Year',
    values='Length_min',
    aggfunc='sum'
    ).sort_index()

years_pivot = years_pivot.assign(
    Pct = lambda x: (x['Length_min'] / total_min * 100)
)

display(years_pivot.round(2))

,Length_min,Length_hours,Length_days,Length_months,Length_years,Pct
Platform Main,,,,,,
ANDROID,213272.17,3554.54,148.11,4.94,0.41,48.41
WINDOWS,125219.27,2086.99,86.96,2.90,0.24,28.42
IOS,80889.34,1348.16,56.17,1.87,0.15,18.36
WEB PLAYER,18144.44,302.41,12.60,0.42,0.03,4.12
PLAYSTATION,1578.20,26.30,1.10,0.04,0.00,0.36
CAST,1114.75,18.58,0.77,0.03,0.00,0.25
UNKNOWN,216.40,3.61,0.15,0.01,0.00,0.05
ANDROID TV,162.54,2.71,0.11,0.00,0.00,0.04


,Length_min,Pct
Year,,
2014,4287.43,0.97
2019,39495.21,8.96
2020,54685.14,12.41
2021,71340.20,16.19
2022,60409.64,13.71
2023,39844.42,9.04
2024,55201.41,12.53
2025,94265.88,21.40
2026,21067.78,4.78
